# Atmospheric CO2 Exploratory Data Analysis

This notebook uses the real `statsmodels` CO2 dataset. Core report generation stays in Python modules so the same analysis runs from Jupyter, the CLI, and CI.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_co2 import load_co2_dataset
from src.features.preprocess_timeseries import build_monthly_features
from src.eda.generate_eda import main as generate_eda

raw = load_co2_dataset()
monthly, _ = build_monthly_features(raw)
series = monthly['co2']

## Dataset shape, date range, and missing values

The source is weekly atmospheric CO2. Monthly preprocessing interpolates missing observations after resampling.

In [ ]:
pd.Series({
    'weekly_rows': len(raw),
    'weekly_start': raw.index.min(),
    'weekly_end': raw.index.max(),
    'missing_weekly_values': int(raw['co2'].isna().sum()),
    'monthly_rows': len(monthly),
    'missing_monthly_values': int(series.isna().sum()),
}, name='value').to_frame()

## Trend and rolling statistics

The first figure shows the long-run CO2 trend. The second compares the monthly series with its trailing 12-month mean and standard deviation.

In [ ]:
generate_eda()
figures = PROJECT_ROOT / 'reports' / 'figures'
display(Image(filename=str(figures / 'co2_timeseries.png')))
display(Image(filename=str(figures / 'rolling_statistics.png')))

## Seasonal pattern and decomposition

Month-of-year averages summarize the annual cycle. Additive decomposition separates observed level, trend, seasonal component, and residuals.

In [ ]:
seasonal_profile = series.groupby(series.index.month).mean().rename_axis('month')
display(seasonal_profile.to_frame('mean_co2_ppm').round(3))
display(Image(filename=str(figures / 'seasonal_decomposition.png')))

## Autocorrelation and stationarity

Strong autocorrelation reflects trend and seasonality. Augmented Dickey-Fuller results compare the level series with its first difference.

In [ ]:
display(Image(filename=str(figures / 'autocorrelation.png')))
level_stat, level_p, *_ = adfuller(series)
diff_stat, diff_p, *_ = adfuller(series.diff().dropna())
pd.DataFrame({
    'series': ['level', 'first difference'],
    'adf_statistic': [level_stat, diff_stat],
    'p_value': [level_p, diff_p],
    'stationary_at_5_percent': [level_p < 0.05, diff_p < 0.05],
}).set_index('series').round(4)

## Interpretation

- CO2 rises over the full observation period and repeats a strong annual cycle.
- The level series is non-stationary; first differencing materially improves stationarity.
- Random splitting would leak future structure, so downstream evaluation uses chronological splits.
- Statistical models are credible benchmarks because the dataset is small, smooth, and strongly seasonal.

The generated narrative is available in `reports/eda_summary.md`.